# AAL Sales Analysis · 01: Inspect the source data

**Course:** Applied Data Science with Python | **Period:** Q4 2020

**Learning goals:** inspect the raw columns, test missing values and duplicates, identify whole-day gaps, and build the first grouped sales summary.

We never overwrite the source dataset. Currency is not specified in the supplied brief, so figures are referred to as **sales values**, not assumed AUD.

## 1. Load the CSV

Save the supplied ZIP (or extracted CSV) in `aal-sales-analysis/data/`. Open this notebook from `notebooks/` or the repository root.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

locations = [
    Path('../data/1739523823_ausapparalsales4thqrt2020.zip'),
    Path('../data/AusApparalSales4thQrt2020.csv'),
    Path('data/1739523823_ausapparalsales4thqrt2020.zip'),
    Path('data/AusApparalSales4thQrt2020.csv'),
]
source_path = next((p for p in locations if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError('Place the supplied ZIP or CSV in the data/ directory first.')
if source_path.suffix.lower() == '.zip':
    with zipfile.ZipFile(source_path) as archive:
        members = [n for n in archive.namelist() if n.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError('Expected exactly one CSV file in the ZIP.')
        with archive.open(members[0]) as stream:
            raw_df = pd.read_csv(stream)
else:
    raw_df = pd.read_csv(source_path)

print('Source:', source_path)
print('Shape:', raw_df.shape)
display(raw_df.head())

## 2. Understand the six source fields

The dataset contains `Date`, `Time`, `State`, `Group`, `Unit`, and `Sales`. Use `isna()` and `notna()` before choosing a treatment for missing data.

In [ ]:
display(raw_df.dtypes.to_frame('dtype'))
display(raw_df.isna().sum().to_frame('Missing: isna()'))
display(raw_df.notna().sum().to_frame('Present: notna()'))

## 3. Clean on a copy and test validity

Whitespace after the CSV delimiters means several text values contain leading spaces. Strip it, parse dates explicitly, and coerce invalid numbers to missing so they can be audited. One expected record is a unique combination of date, time, state, and group.

In [ ]:
df = raw_df.copy()
for column in ['Time', 'State', 'Group']:
    df[column] = df[column].astype('string').str.strip()
df['Date'] = pd.to_datetime(df['Date'].astype('string').str.strip(), format='%d-%b-%Y', errors='coerce')
for column in ['Unit', 'Sales']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Missing or invalid after parsing:')
display(df.isna().sum().to_frame('Count'))

checks = pd.Series({
    'Exact duplicate rows': int(df.duplicated().sum()),
    'Duplicate date/time/state/group keys': int(df.duplicated(['Date', 'Time', 'State', 'Group']).sum()),
    'Units <= 0': int((df['Unit'] <= 0).sum()),
    'Sales < 0': int((df['Sales'] < 0).sum()),
    'Non-integer units': int(((df['Unit'] % 1 != 0) & df['Unit'].notna()).sum()),
})
display(checks.to_frame('Count'))

## 4. Audit date completeness

No null cells does not mean a complete calendar. Compare observed days with all **92** dates in Q4 2020.

In [ ]:
quarter = pd.date_range('2020-10-01', '2020-12-31', freq='D')
observed = pd.DatetimeIndex(df['Date'].dropna().unique())
missing_dates = quarter.difference(observed)
print('Days observed:', len(observed), '/', len(quarter))
print('Missing days:', [d.strftime('%d %b %Y') for d in missing_dates])
print('Rows per recorded day:', df.groupby('Date').size().agg(['min', 'max']).to_dict())
print('Rows outside Q4:', int((~df['Date'].between(quarter.min(), quarter.max())).sum()))

## 5. Explore the relationship between units and sales

Determine whether `Sales` contains independent pricing information or is simply unit volume multiplied by a fixed amount.

In [ ]:
print('Distinct Sales / Unit ratios:', sorted((df['Sales'] / df['Unit']).dropna().unique().tolist()))
print('Rows disagreeing with Sales = 2500 × Unit:', int((df['Sales'] != 2500 * df['Unit']).sum()))
display(df[['Unit', 'Sales']].describe().round(2))
print('Most common unit count:', df['Unit'].mode().tolist())

## 6. Your first GroupBy

Aggregate sales by state, demographic group, time of day and month. This is also a good place to check that records are distributed evenly across states.

In [ ]:
state_sales = (df.groupby('State', as_index=False)
               .agg(Sales=('Sales', 'sum'), Units=('Unit', 'sum'), Records=('Sales', 'size'))
               .sort_values('Sales', ascending=False))
state_sales['Share_pct'] = 100 * state_sales['Sales'] / state_sales['Sales'].sum()
group_sales = df.groupby('Group', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
time_sales = df.groupby('Time', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
monthly_sales = (df.assign(Month=df['Date'].dt.to_period('M').astype('string'))
                 .groupby('Month', as_index=False)['Sales'].sum())

display(state_sales)
display(group_sales)
display(time_sales)
display(monthly_sales)

## 7. Visualise state sales

The brief recommends Seaborn for statistical plots. It works with Pandas tables, while Matplotlib offers fine control over plot labels.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=state_sales, y='State', x='Sales', color='steelblue', errorbar=None, ax=ax)
ax.set(title='Recorded sales by state/territory | Q4 2020',
       xlabel='Sales (currency unspecified)', ylabel='State/territory')
ax.ticklabel_format(axis='x', style='plain')
plt.tight_layout()
plt.show()

## Findings and limitations

- 7,560 source rows and six columns; zero nulls and zero duplicate date/time/state/group keys.
- Only 90 of 92 Q4 dates appear. **31 October and 31 December 2020 are absent**; do not invent replacement sales.
- Every row satisfies `Sales = 2,500 × Unit`; unit and sales rankings are mathematically equivalent.
- Victoria has the highest recorded total and Western Australia the lowest, but we lack store count, local demand and population denominators.
- The source has no ACT observations and does not define its currency.

**Next step:** descriptive statistics, a documented missing-data decision, Min–Max normalization, weekly/monthly/quarterly reports, and the required dashboard.